# Panel Cointegration Tests - SOLUTIONS

**Version**: 1.0  
**Date**: 2026-02-22

Solutions for Exercises 1-3 from Notebook 02: Panel Cointegration Tests.

In [ ]:
%matplotlib inline

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")

# Local utilities
import sys

import statsmodels.api as sm
from scipy import stats

# PanelBox imports
import panelbox
from panelbox.diagnostics.cointegration import kao_test, pedroni_test, westerlund_test
from panelbox.diagnostics.unit_root import panel_unit_root_test

BASE_DIR = Path("..")
sys.path.insert(0, str(BASE_DIR / "utils"))
from cointegration_helpers import compare_cointegration_methods
from unit_root_helpers import compare_unit_root_tests

# Configuration
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["font.size"] = 11
pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 4)
np.random.seed(42)

# Paths
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs"
FIGURES_DIR = OUTPUT_DIR / "figures" / "cointegration"
TABLES_DIR = OUTPUT_DIR / "tables" / "cointegration"
RESULTS_DIR = OUTPUT_DIR / "results" / "cointegration"

for d in [FIGURES_DIR, TABLES_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Setup complete!")
print(f"PanelBox version: {panelbox.__version__}")

---

## Exercise 1: Interest Rate Parity (Easy)

**Task:** Test covered interest rate parity using international interest rates.

**Theory:** $i_t - i^*_t = f_t - s_t$, i.e., the interest rate spread should be cointegrated with the forward premium.

In [ ]:
print("=" * 80)
print("EXERCISE 1: INTEREST RATE PARITY")
print("=" * 80)

# Step 1: Load data
ir_data = pd.read_csv(DATA_DIR / "cointegration" / "interest_rates.csv")

print(f"\nDataset shape: {ir_data.shape}")
print(f"Countries: {ir_data['country'].nunique()}")
print(f"Years: {ir_data['year'].min()} - {ir_data['year'].max()}")
print(f"Columns: {list(ir_data.columns)}")
display(ir_data.head(10))

In [ ]:
# Step 2: Verify variables are I(1)
print("Step 2: Unit Root Tests")
print("=" * 60)

for var_name in ["spread", "forward_premium"]:
    print(f"\nTesting: {var_name}")
    print("-" * 40)
    try:
        ur_result = panel_unit_root_test(
            ir_data, var_name, entity_col="country", time_col="year", test="all", trend="c"
        )
        print(ur_result.summary_table())
    except Exception as e:
        print(f"  Error: {e}")
        # Fallback: use compare_unit_root_tests
        try:
            comp = compare_unit_root_tests(ir_data, var_name, "country", "year", trend="c")
            display(comp)
        except Exception as e2:
            print(f"  Fallback also failed: {e2}")

In [ ]:
# Step 3: Test cointegration using Pedroni
print("Step 3: Pedroni Cointegration Test")
print("=" * 60)
print("H0: No cointegration between spread and forward_premium\n")

try:
    irp_pedroni = pedroni_test(
        data=ir_data,
        entity_col="country",
        time_col="year",
        y_var="spread",
        x_vars=["forward_premium"],
        method="all",
        trend="c",
    )

    print(irp_pedroni.summary())

    # Count rejections
    rejections = irp_pedroni.reject_at(alpha=0.05)
    if isinstance(rejections, dict):
        n_reject = sum(1 for v in rejections.values() if v)
        n_total = len(rejections)
        print(f"\nRejections at 5%: {n_reject} / {n_total}")

except Exception as e:
    print(f"Pedroni test error: {e}")

In [ ]:
# Step 4: Interpretation
print("Step 4: Economic Interpretation")
print("=" * 60)

# Also test with Kao for comparison
try:
    irp_kao = kao_test(
        data=ir_data,
        entity_col="country",
        time_col="year",
        y_var="spread",
        x_vars=["forward_premium"],
        method="all",
    )
    print("Kao test results:")
    print(irp_kao.summary())
except Exception as e:
    print(f"Kao test error: {e}")

# Visualization
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
sample_countries = sorted(ir_data["country"].unique())[:6]

for i, country in enumerate(sample_countries):
    cdata = ir_data[ir_data["country"] == country].sort_values("year")
    ax = axes[i]
    ax.plot(cdata["year"], cdata["spread"], label="Spread (i - i*)", linewidth=1.2)
    ax.plot(
        cdata["year"],
        cdata["forward_premium"],
        label="Forward premium",
        linewidth=1.2,
        linestyle="--",
    )
    ax.set_title(country, fontweight="bold")
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)

plt.suptitle("Interest Rate Parity: Spread vs Forward Premium", fontweight="bold", fontsize=14)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "02_ex1_irp.png", dpi=300, bbox_inches="tight")
plt.show()

print("\nInterpretation:")
print("If cointegration is found, Covered Interest Rate Parity holds in the long run.")
print("Short-run deviations (due to transaction costs, risk premia) are temporary.")

---

## Exercise 2: Compare Methods (Medium)

**Task:** Apply all three methods (Pedroni, Westerlund, Kao) to the OECD consumption-income data with different trend specifications.

In [ ]:
print("=" * 80)
print("EXERCISE 2: COMPARE METHODS")
print("=" * 80)

# Load data
oecd = pd.read_csv(DATA_DIR / "cointegration" / "oecd_macro.csv")
print(f"Data: {oecd.shape[0]} obs, {oecd['country'].nunique()} countries\n")

In [ ]:
# Step 1: Run with trend='c' (constant only)
print('Test with trend="c" (constant only)')
print("=" * 60)

comp_c = compare_cointegration_methods(
    data=oecd,
    y_var="log_C",
    x_vars=["log_Y"],
    entity_col="country",
    time_col="year",
    save_path=str(TABLES_DIR / "02_ex2_comparison_c.csv"),
)

display(comp_c)

if "Reject_5pct" in comp_c.columns:
    n_reject_c = comp_c["Reject_5pct"].sum()
    n_total = len(comp_c)
    print(f"\nTotal rejections (trend=c): {n_reject_c} / {n_total}")

In [ ]:
# Step 2: Run individual tests with trend='ct' for comparison
print('Test with trend="ct" (constant + trend)')
print("=" * 60)

rows_ct = []

# Pedroni with ct
try:
    pedroni_ct = pedroni_test(
        data=oecd,
        entity_col="country",
        time_col="year",
        y_var="log_C",
        x_vars=["log_Y"],
        method="all",
        trend="ct",
    )
    stat = pedroni_ct.statistic
    pval = pedroni_ct.pvalue
    if isinstance(stat, dict):
        for test_name in stat:
            s = stat[test_name]
            p = pval.get(test_name, np.nan) if isinstance(pval, dict) else pval
            rows_ct.append(
                {
                    "Method": "Pedroni",
                    "Test": test_name,
                    "Statistic": s,
                    "P-value": p,
                    "Reject_5pct": p < 0.05 if not np.isnan(p) else None,
                }
            )
except Exception as e:
    print(f"Pedroni ct error: {e}")

# Westerlund with ct
try:
    west_ct = westerlund_test(
        data=oecd,
        entity_col="country",
        time_col="year",
        y_var="log_C",
        x_vars=["log_Y"],
        method="all",
        trend="ct",
        use_bootstrap=False,
    )
    stat = west_ct.statistic
    pval = west_ct.pvalue
    if isinstance(stat, dict):
        for test_name in stat:
            s = stat[test_name]
            p = pval.get(test_name, np.nan) if isinstance(pval, dict) else pval
            rows_ct.append(
                {
                    "Method": "Westerlund",
                    "Test": test_name,
                    "Statistic": s,
                    "P-value": p,
                    "Reject_5pct": p < 0.05 if not np.isnan(p) else None,
                }
            )
except Exception as e:
    print(f"Westerlund ct error: {e}")

# Kao with ct
try:
    kao_ct = kao_test(
        data=oecd,
        entity_col="country",
        time_col="year",
        y_var="log_C",
        x_vars=["log_Y"],
        method="all",
        trend="ct",
    )
    stat = kao_ct.statistic
    pval = kao_ct.pvalue
    if isinstance(stat, dict):
        for test_name in stat:
            s = stat[test_name]
            p = pval.get(test_name, np.nan) if isinstance(pval, dict) else pval
            rows_ct.append(
                {
                    "Method": "Kao",
                    "Test": test_name,
                    "Statistic": s,
                    "P-value": p,
                    "Reject_5pct": p < 0.05 if not np.isnan(p) else None,
                }
            )
except Exception as e:
    print(f"Kao ct error: {e}")

comp_ct = pd.DataFrame(rows_ct)
display(comp_ct)

if "Reject_5pct" in comp_ct.columns and len(comp_ct) > 0:
    n_reject_ct = comp_ct["Reject_5pct"].sum()
    n_total_ct = len(comp_ct)
    print(f"\nTotal rejections (trend=ct): {n_reject_ct} / {n_total_ct}")

comp_ct.to_csv(TABLES_DIR / "02_ex2_comparison_ct.csv", index=False)

In [ ]:
# Step 3: Compare and discuss
print("Step 3: Discussion")
print("=" * 60)

print("\n1. Do all three methods agree on cointegration?")
if "Reject_5pct" in comp_c.columns:
    for method in comp_c["Method"].unique():
        subset = comp_c[comp_c["Method"] == method]
        n_r = subset["Reject_5pct"].sum()
        n_t = len(subset)
        print(f"   {method} (trend=c): {n_r}/{n_t} rejections")

print("\n2. Which method rejects most strongly?")
if "P-value" in comp_c.columns:
    min_p = comp_c.loc[comp_c["P-value"].idxmin()] if len(comp_c) > 0 else None
    if min_p is not None:
        print(f"   Lowest p-value: {min_p['Method']} {min_p['Test']} (p={min_p['P-value']:.4f})")

print("\n3. How do results change with trend specification?")
if "Reject_5pct" in comp_c.columns and "Reject_5pct" in comp_ct.columns:
    n_c = comp_c["Reject_5pct"].sum() if len(comp_c) > 0 else 0
    n_ct = comp_ct["Reject_5pct"].sum() if len(comp_ct) > 0 else 0
    print(f"   Rejections with trend=c: {n_c}")
    print(f"   Rejections with trend=ct: {n_ct}")
    if n_c > n_ct:
        print("   -> trend=c rejects more often (adding trend reduces power)")
    elif n_ct > n_c:
        print("   -> trend=ct rejects more often (trend specification matters)")
    else:
        print("   -> Similar results regardless of trend specification")

print("\nConclusion:")
print("  Consumption and income appear cointegrated across OECD countries.")
print("  This is consistent with the Permanent Income Hypothesis.")
print("  Multiple tests provide robustness to the finding.")

---

## Exercise 3: Heterogeneity Analysis (Hard)

**Task:** Investigate heterogeneity in cointegration vectors using country-specific OLS regressions.

In [ ]:
print("=" * 80)
print("EXERCISE 3: HETEROGENEITY ANALYSIS")
print("=" * 80)

# Step 1: Estimate OLS cointegrating regression per country
print("\nStep 1: OLS per country: log(C) = alpha_i + beta_i * log(Y) + epsilon")
print("-" * 60)

oecd = pd.read_csv(DATA_DIR / "cointegration" / "oecd_macro.csv")

country_results = []
for country in sorted(oecd["country"].unique()):
    cdata = oecd[oecd["country"] == country].sort_values("year")
    X = sm.add_constant(cdata["log_Y"].values)
    ols = sm.OLS(cdata["log_C"].values, X).fit()
    country_results.append(
        {
            "Country": country,
            "Beta (MPC)": ols.params[1],
            "Alpha": ols.params[0],
            "SE_Beta": ols.bse[1],
            "t_stat": ols.tvalues[1],
            "R_squared": ols.rsquared,
        }
    )

beta_df = pd.DataFrame(country_results)
display(beta_df)

print(f"\nAverage beta: {beta_df['Beta (MPC)'].mean():.4f}")
print(f"Std of betas: {beta_df['Beta (MPC)'].std():.4f}")
print(f"Range: [{beta_df['Beta (MPC)'].min():.4f}, {beta_df['Beta (MPC)'].max():.4f}]")

In [ ]:
# Step 2: Group countries by median beta
print("Step 2: Group Analysis")
print("=" * 60)

median_beta = beta_df["Beta (MPC)"].median()
beta_df["Group"] = np.where(beta_df["Beta (MPC)"] > median_beta, "High MPC", "Low MPC")

print(f"Median beta: {median_beta:.4f}")
print(f"\nHigh MPC group (beta > {median_beta:.4f}):")
high_group = beta_df[beta_df["Group"] == "High MPC"]
display(high_group[["Country", "Beta (MPC)"]].sort_values("Beta (MPC)", ascending=False))

print(f"\nLow MPC group (beta <= {median_beta:.4f}):")
low_group = beta_df[beta_df["Group"] == "Low MPC"]
display(low_group[["Country", "Beta (MPC)"]].sort_values("Beta (MPC)", ascending=False))

In [ ]:
# Step 3: Test if average beta differs between groups (t-test)
print("Step 3: Test for Group Differences")
print("=" * 60)

high_betas = high_group["Beta (MPC)"].values
low_betas = low_group["Beta (MPC)"].values

t_stat, p_value = stats.ttest_ind(high_betas, low_betas)

print(f"\nHigh MPC group: mean={high_betas.mean():.4f}, n={len(high_betas)}")
print(f"Low MPC group:  mean={low_betas.mean():.4f}, n={len(low_betas)}")
print("\nt-test for difference in means:")
print(f"  t-statistic: {t_stat:.4f}")
print(f"  p-value: {p_value:.4f}")
print(f"  Significant at 5%: {p_value < 0.05}")

if p_value < 0.05:
    print("\n-> The MPC differs significantly between groups.")
    print("   This suggests heterogeneous cointegration vectors.")
else:
    print("\n-> No significant difference between groups.")
    print("   Homogeneous cointegration may be a reasonable assumption.")

In [ ]:
# Step 4: Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: bar chart of country-specific betas
ax = axes[0]
beta_sorted = beta_df.sort_values("Beta (MPC)")
colors = ["steelblue" if g == "Low MPC" else "coral" for g in beta_sorted["Group"]]
ax.barh(
    beta_sorted["Country"],
    beta_sorted["Beta (MPC)"],
    color=colors,
    edgecolor="black",
    linewidth=0.5,
)
ax.axvline(
    median_beta, color="green", linestyle="--", linewidth=2, label=f"Median={median_beta:.3f}"
)
ax.axvline(1.0, color="black", linestyle=":", linewidth=1.5, label="$\\beta=1$")
ax.set_xlabel("$\\beta_i$ (Marginal Propensity to Consume)")
ax.set_title("Country-Specific Cointegrating Coefficients", fontweight="bold")
ax.legend(fontsize=9)

# Right: box plot by group
ax = axes[1]
group_data = [high_betas, low_betas]
bp = ax.boxplot(group_data, labels=["High MPC", "Low MPC"], patch_artist=True)
bp["boxes"][0].set_facecolor("coral")
bp["boxes"][1].set_facecolor("steelblue")
ax.axhline(1.0, color="black", linestyle=":", linewidth=1.5, label="$\\beta=1$")
ax.set_ylabel("$\\beta_i$ (MPC)")
ax.set_title(f"Group Comparison (t={t_stat:.2f}, p={p_value:.3f})", fontweight="bold")
ax.legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / "02_ex3_heterogeneity.png", dpi=300, bbox_inches="tight")
plt.show()

# Save results
beta_df.to_csv(TABLES_DIR / "02_ex3_country_betas.csv", index=False)

print("\nKey findings:")
print(
    f"1. Country-specific MPC ranges from {beta_df['Beta (MPC)'].min():.3f} to {beta_df['Beta (MPC)'].max():.3f}"
)
print(f"2. Average MPC = {beta_df['Beta (MPC)'].mean():.3f} (consistent with PIH: MPC < 1)")
print(f"3. Group difference p-value = {p_value:.3f}")
print(
    "4. Heterogeneity is present -> group statistics (Pedroni) are preferred over panel statistics"
)

---

## Summary

| Exercise | Finding |
|----------|--------|
| **1. IRP** | Interest rate spread and forward premium are cointegrated $\rightarrow$ IRP holds long-run |
| **2. Compare** | All three methods generally agree; Pedroni is most comprehensive |
| **3. Heterogeneity** | MPC varies across countries; group statistics are appropriate |